# ⚽ World Cup 2026 — Match Predictor (Poisson + Elo)

Predicting knockout matches at the FIFA World Cup 2026 using a **Poisson goals model** combined with **Elo ratings**, trained on international match results since 2018.

**Example predicted:** Egypt 🇪🇬 vs Argentina 🇦🇷 — Round of 16.

---
### How it works
1. **Attack / Defense strength** — how many goals each team scores & concedes vs the league average.
2. **Poisson distribution** — turns those rates into probabilities for every scoreline.
3. **Elo ratings** — measures overall strength while accounting for opponent quality.
4. **Knockout scenario** — extra time & penalty probabilities (knockout games can't end in a draw).

**Data:** [International football results (1872–2026)](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)


## 1. Load & clean the data

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import zipfile

# Path to the Kaggle dataset zip
zip_path = 'International football results from 1872 to 2026.zip'

with zipfile.ZipFile(zip_path) as z:
    with z.open('results.csv') as f:
        df = pd.read_csv(f, parse_dates=['date'])

# Separate unplayed matches (our prediction targets) from played ones
future = df[df['home_score'].isna()].copy()

# Keep only played matches — the model learns from real results
df = df.dropna(subset=['home_score', 'away_score']).copy()
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

print("Played matches:", len(df))

## 2. Focus on recent form

Matches from decades ago tell us nothing about today's teams. We keep only matches since **2018** to capture *current* strength, and compute the league-wide average goals as our baseline.

In [ ]:
CUTOFF = '2018-01-01'
recent = df[df['date'] >= CUTOFF].copy()

overall_avg = (recent['home_score'].mean() + recent['away_score'].mean()) / 2
print("Matches since", CUTOFF, ":", len(recent))
print("Average goals per team per match:", round(overall_avg, 3))

## 3. Attack & Defense strength

For each team, relative to the league average (where **1.0 = average**):
- **Attack** > 1 means scores more than average.
- **Defense** < 1 means concedes fewer than average (lower is better).

In [ ]:
teams = pd.unique(recent[['home_team', 'away_team']].values.ravel())

rows = []
for t in teams:
    home_games = recent[recent['home_team'] == t]
    away_games = recent[recent['away_team'] == t]
    games = len(home_games) + len(away_games)
    if games < 5:                      # skip tiny samples
        continue
    goals_for     = home_games['home_score'].sum() + away_games['away_score'].sum()
    goals_against = home_games['away_score'].sum() + away_games['home_score'].sum()
    rows.append({
        'team': t,
        'games': games,
        'attack':  (goals_for / games) / overall_avg,
        'defense': (goals_against / games) / overall_avg,
    })

S = pd.DataFrame(rows).set_index('team')
S.loc[['Egypt', 'Argentina']]

## 4. Elo ratings

Elo starts every team at 1500 and updates after each match. Beating a strong team earns more points than beating a weak one — so it accounts for **opponent quality**, which raw goal counts ignore.

In [ ]:
elo = {t: 1500.0 for t in teams}
K = 30

for _, r in recent.sort_values('date').iterrows():
    h, a = r['home_team'], r['away_team']
    Rh, Ra = elo[h], elo[a]

    neutral = (r['neutral'] == True) or (str(r['neutral']).upper() == 'TRUE')
    home_adv = 0 if neutral else 60

    Eh = 1 / (1 + 10 ** ((Ra - (Rh + home_adv)) / 400))

    if   r['home_score'] > r['away_score']: Sh = 1
    elif r['home_score'] < r['away_score']: Sh = 0
    else:                                   Sh = 0.5

    gd = abs(r['home_score'] - r['away_score'])
    mult = 1 if gd <= 1 else (1.5 if gd == 2 else (1.75 if gd == 3 else 2))

    elo[h] = Rh + K * mult * (Sh - Eh)
    elo[a] = Ra + K * mult * ((1 - Sh) - (1 - Eh))

print("Egypt Elo:    ", round(elo['Egypt']))
print("Argentina Elo:", round(elo['Argentina']))

## 5. Predict the match (Poisson)

Expected goals = league average × team attack × opponent defense. Neutral ground (World Cup), so no home advantage.

In [ ]:
def predict(home, away, neutral=True, max_goals=10):
    la_h = overall_avg * S.loc[home, 'attack'] * S.loc[away, 'defense']
    la_a = overall_avg * S.loc[away, 'attack'] * S.loc[home, 'defense']
    if not neutral:
        la_h *= 1.1

    m = np.outer(poisson.pmf(range(max_goals), la_h),
                 poisson.pmf(range(max_goals), la_a))
    return la_h, la_a, m

la_egy, la_arg, m = predict('Egypt', 'Argentina', neutral=True)

p_egy = np.tril(m, -1).sum()
p_draw = np.trace(m)
p_arg = np.triu(m, 1).sum()

print(f"Expected goals -> Egypt: {la_egy:.2f} | Argentina: {la_arg:.2f}")
print(f"Egypt win: {p_egy*100:.1f}%  |  Draw: {p_draw*100:.1f}%  |  Argentina win: {p_arg*100:.1f}%")

## 6. Knockout scenario — extra time & penalties

A Round of 16 match can't end in a draw. If tied after 90 minutes → 30 min extra time (goal rate ÷ 3). Still tied → penalties (treated as a 50/50 coin flip).

In [ ]:
la_e_et, la_a_et = la_egy / 3, la_arg / 3
m_et = np.outer(poisson.pmf(range(10), la_e_et),
                poisson.pmf(range(10), la_a_et))
p_e_et = np.tril(m_et, -1).sum()
p_d_et = np.trace(m_et)
p_a_et = np.triu(m_et, 1).sum()

goes_et   = p_draw
goes_pens = p_draw * p_d_et

final_egy = p_egy + p_draw * (p_e_et + p_d_et * 0.5)
final_arg = p_arg + p_draw * (p_a_et + p_d_et * 0.5)

print(f"Decided in 90 min:  {(p_egy + p_arg)*100:.1f}%")
print(f"Goes to extra time: {goes_et*100:.1f}%")
print(f"Goes to penalties:  {goes_pens*100:.1f}%")
print()
print(f"Egypt qualify:     {final_egy*100:.1f}%")
print(f"Argentina qualify: {final_arg*100:.1f}%")

## 7. Export results for Power BI

Save clean CSVs to build an interactive dashboard.

In [ ]:
pd.DataFrame([{
    'match': 'Egypt vs Argentina',
    'egypt_win_pct': round(p_egy*100, 1),
    'draw_pct': round(p_draw*100, 1),
    'argentina_win_pct': round(p_arg*100, 1),
    'goes_to_penalties_pct': round(goes_pens*100, 1),
    'egypt_qualify_pct': round(final_egy*100, 1),
    'argentina_qualify_pct': round(final_arg*100, 1),
}]).to_csv('prediction.csv', index=False)

S.reset_index().to_csv('team_strengths.csv', index=False)
print("Saved prediction.csv and team_strengths.csv")

---
### ⚠️ Note
Football is unpredictable — this model estimates probabilities from historical form, not certainties. Injuries, form, and tournament pressure all play a role. Built for learning and fun. ⚽
